# Solutions · Chapter 04-04 · Splitting II: grouped and chronological

Worked answers for `notebooks/04_workflow/04-04_splitting_group_time.ipynb`.

E9, E10 and E12 all qualify the chapter's headline in useful ways. E20 finds a fourth kind of leak that
neither grouping nor chronology removes.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")


# SYNTHETIC. The gym panel from 04-01, stacked over every month as in 04-04.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members().sort_values(["member_id", "month"])


def frame_at(cut, horizon=6):
    active = panel[(panel.month == cut) & (panel.cancelled == 0)].member_id.unique()
    if len(active) == 0:
        return None
    history = panel[panel.member_id.isin(active) & (panel.month <= cut)]
    future = panel[panel.member_id.isin(active) & (panel.month > cut)]
    left = future[(future.month <= cut + horizon) & (future.cancelled == 1)].member_id.unique()
    by_member = history.groupby("member_id")
    frame = pd.DataFrame({
        "visits_now": history[history.month == cut].set_index("member_id").visits.reindex(active),
        "mean_visits_last_3": history[history.month > cut - 3].groupby("member_id").visits.mean().reindex(active),
        "lifetime_mean_visits": by_member.visits.mean().reindex(active),
        "lifetime_sd_visits": by_member.visits.std().reindex(active).fillna(0),
        "join_month": by_member.month.min().reindex(active),
        "tenure_months": by_member.size().reindex(active),
        "tickets_so_far": by_member.tickets.sum().reindex(active),
    })
    frame["member_id"] = active
    frame["cut_month"] = cut
    frame["target"] = np.isin(active, left).astype(int)
    return frame.reset_index(drop=True)


stacked = pd.concat([frame_at(cut) for cut in range(1, 19)], ignore_index=True)
FEATURES = ["visits_now", "mean_visits_last_3", "lifetime_mean_visits", "lifetime_sd_visits",
            "join_month", "tenure_months", "tickets_so_far"]
X, y, groups = stacked[FEATURES], stacked.target, stacked.member_id


def forest_on(train_rows, test_rows):
    model = RandomForestClassifier(n_estimators=300, random_state=0, min_samples_leaf=1)
    model.fit(X.iloc[train_rows], y.iloc[train_rows])
    return roc_auc_score(y.iloc[test_rows], model.predict_proba(X.iloc[test_rows])[:, 1])


held_out_members = stacked.member_id % 5 == 0
print("%d rows, %d members" % (len(stacked), stacked.member_id.nunique()))

## Quick understanding

### E1

**Question 1: does the same entity produce more than one row?**
**Question 2: will the model predict the future from the past?**

| | time does not matter | time matters |
|---|---|---|
| **rows independent** | stratified random split | chronological split or forward chaining |
| **entities repeat** | `GroupKFold` | grouped **and** forward - the strictest |

### E2

Because k-nearest-neighbours **is** memory: it predicts by finding the most similar training rows. For
member 314's month 10, the five most similar rows in the entire dataset are that same member's months 7,
8, 9, 11 and 12 - nearly identical feature vectors, often with the same label. Under a random split those
rows are sitting in the training set, so kNN is doing a lookup rather than a prediction.

Logistic regression has seven coefficients and no per-row storage. The only thing it can express is a
smooth trend across everybody, so having seen member 314 before buys it almost nothing: **+0.0021**
against kNN's **+0.1052**.

### E3

Because an inflated score disappoints you once, in production, and you recalibrate - whereas a reversed
ranking makes you **ship the wrong model**, and every future comparison is then made against the wrong
incumbent.

## Hand calculation

### E4

Each of a patient's 10 rows independently goes to training with probability 0.8.

**(a)** No rows in training: `0.2^10` = **1.02e-07**. Expected number of genuinely unseen patients among
100: `100 x 1.02e-07` = **0.00001**. Effectively zero - it will essentially never happen.

**(b)** No rows in the test set: `0.8^10` = **0.107**, so about **11 of the 100 patients** appear only in
training.

**Together:** roughly 89 patients have rows on both sides, 11 are training-only, and none are test-only.
**The test set contains no new patients at all.** It is measuring how well the model predicts new
*scans* from patients it has already studied - which is a real question, but it is not the question a
hospital deploying the model is asking.

### E5

From the ladder: random rows 0.7580, grouped only 0.6624, chronological only 0.7124.

- Attributable to grouping: `0.7580 - 0.6624` = **0.0956**
- Attributable to chronology: `0.7580 - 0.7124` = **0.0456**
- The two together: **0.1412**, against a total drop to column 4 of `0.7580 - 0.6446` = **0.1134**

**They do not add up, and they are not supposed to.** Columns 2 and 3 each remove one leak *while the
other is still present*, so both measurements include some of the same shared advantage - the part of the
score that grouping and chronology were both exploiting gets counted twice. Removing the second leak,
once the first is gone, is worth less than removing it first.

That is a general property of overlapping causes, and the practical rule is: **measure the effect of each
fix in the presence of the other fixes**, not in isolation, or your attributions will over-explain.

### E6

Mean **0.7590**, standard deviation **0.0082**.

**What the standard deviation tells you:** the five folds agree closely. If your only concern were
sampling noise, you would be entitled to quote 0.759 with confidence.

**What it does not tell you:** anything at all about the 0.096 of inflation. Every fold used the same
leaky splitting rule, so every fold is wrong in the same direction by roughly the same amount. **A small
fold-to-fold spread measures consistency, not correctness**, and a leaky procedure is consistently wrong -
which makes it look *more* trustworthy, not less. This is the most dangerous property of leakage: it
raises the mean and lowers the variance at the same time.

### E7

Forward chaining with walls at 8, 10, 12, 14, 16 and a two-month window tests months 9-10, 11-12, 13-14,
15-16 and 17-18: **10 distinct test months**, and every month from 9 to 18 is tested exactly once.

A single chronological split at month 12 tests months 13-18: **6 distinct test months**, and trains on
months 1-12.

**Forward chaining uses more of the data as test** (10 months against 6) and gets five estimates instead
of one. **The single split trains on more data in its one fit** (12 months) than three of the five
chaining folds do. The trade is the usual one - forward chaining costs five fits and buys a spread.

In [ ]:
print("E4  P(no training rows)  = 0.2^10 = %.2e  -> %.5f unseen patients of 100" % (0.2 ** 10, 100 * 0.2 ** 10))
print("    P(no test rows)      = 0.8^10 = %.4f  -> %.1f training-only patients of 100" % (0.8 ** 10, 100 * 0.8 ** 10))
print()
print("E5  grouping    : 0.7580 - 0.6624 = %.4f" % (0.7580 - 0.6624))
print("    chronology  : 0.7580 - 0.7124 = %.4f" % (0.7580 - 0.7124))
print("    sum %.4f against an actual total drop of %.4f" % (0.0956 + 0.0456, 0.7580 - 0.6446))
print()
folds = [0.755, 0.762, 0.749, 0.771, 0.758]
print("E6  mean %.4f, sd %.4f  -- and none of it detects the 0.096 of inflation"
      % (np.mean(folds), np.std(folds, ddof=1)))

## Coding

### E8 - the two assertions

In [ ]:
def leak_check(train_frame, test_frame):
    shared = set(train_frame.member_id) & set(test_frame.member_id)
    if shared:
        raise AssertionError("%d members appear on both sides, for example %s"
                             % (len(shared), sorted(shared)[:5]))
    if train_frame.cut_month.max() >= test_frame.cut_month.min():
        raise AssertionError("training reaches month %d but testing starts at month %d"
                             % (train_frame.cut_month.max(), test_frame.cut_month.min()))
    print("clean: %d train rows, %d test rows, no shared members, no time travel"
          % (len(train_frame), len(test_frame)))


for label, train_mask, test_mask in [
    ("random split", stacked.index % 10 < 7, stacked.index % 10 >= 7),
    ("grouped only", ~held_out_members, held_out_members),
    ("grouped AND forward", (stacked.cut_month <= 12) & ~held_out_members,
     (stacked.cut_month > 12) & held_out_members),
]:
    print("--", label)
    try:
        leak_check(stacked[train_mask], stacked[test_mask])
    except AssertionError as failure:
        print("   FAILED:", failure)

Each split fails on exactly the leak it has. **These two assertions belong in the pipeline, not in a
notebook** - they cost microseconds, they fail loudly, and between them they would have caught every
problem in this chapter without anybody needing to remember the lesson.

### E9 - inflation against capacity

In [ ]:
rows = []
for depth in [2, 5, 10, None]:
    def score(train_rows, test_rows, depth=depth):
        model = DecisionTreeClassifier(max_depth=depth, random_state=0)
        model.fit(X.iloc[train_rows], y.iloc[train_rows])
        return roc_auc_score(y.iloc[test_rows], model.predict_proba(X.iloc[test_rows])[:, 1])

    random_mean = np.mean([score(a, b) for a, b in
                           StratifiedKFold(5, shuffle=True, random_state=0).split(X, y)])
    grouped_mean = np.mean([score(a, b) for a, b in GroupKFold(5).split(X, y, groups=groups)])
    leaves = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X, y).get_n_leaves()
    rows.append({"max depth": depth if depth else "unlimited", "leaves": leaves,
                 "random split": round(random_mean, 4), "grouped split": round(grouped_mean, 4),
                 "inflation": round(random_mean - grouped_mean, 4)})
depth_table = pd.DataFrame(rows)
print(depth_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(range(len(depth_table)), depth_table["random split"], "o-", color="#D55E00",
        label="random split")
ax.plot(range(len(depth_table)), depth_table["grouped split"], "s-", color="#0072B2",
        label="grouped split")
ax.fill_between(range(len(depth_table)), depth_table["grouped split"], depth_table["random split"],
                color="#D55E00", alpha=0.15)
ax.set_xticks(range(len(depth_table)))
ax.set_xticklabels(["depth %s\n(%d leaves)" % (r["max depth"], r["leaves"])
                    for _, r in depth_table.iterrows()], fontsize=8)
ax.set_ylabel("AUC")
ax.set_title("The shaded gap is the leak. It opens as the tree gains capacity", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Inflation starts at essentially zero and grows with capacity - up to a point.**

- **depth 2**, 4 leaves: **+0.0019**. Four leaves cannot hold a fact about 580 members.
- **depth 5**, 31 leaves: **+0.0328**.
- **depth 10**, 238 leaves: **+0.0835**. Now there is roughly one leaf per two or three members.
- **unlimited**, 842 leaves: **+0.0607** - *lower*.

The last row is the interesting one and the claim needs qualifying because of it. **Inflation does not
grow without limit; it peaks and then falls**, because by 842 leaves the tree is so overfitted that its
*random-split* score has collapsed too - from 0.6819 to 0.6103. You cannot inflate a score that has
already fallen apart.

So the accurate statement is: **inflation scales with the capacity a model has to memorise entities, over
the range where the model is still working.** Depth 10 is the worst of both worlds here - flexible enough
to memorise, not yet broken enough to look bad on the leaky split.

### E10 - how much was leakage and how much was less data?

In [ ]:
full_folds, subsampled_folds = [], []
subsample_rng = np.random.default_rng(0)
for train_rows, test_rows in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
    full_folds.append(forest_on(train_rows, test_rows))
    smaller = subsample_rng.choice(train_rows, size=2740, replace=False)
    subsampled_folds.append(forest_on(smaller, test_rows))

strict_train = np.where((stacked.cut_month <= 12) & ~held_out_members)[0]
strict_test = np.where((stacked.cut_month > 12) & held_out_members)[0]
strict = forest_on(strict_train, strict_test)

print("random split, ~5,054 training rows : %.4f" % np.mean(full_folds))
print("random split,  2,740 training rows : %.4f" % np.mean(subsampled_folds))
print("grouped AND forward, 2,740 rows    : %.4f" % strict)
print()
print("total drop                 : %.4f" % (np.mean(full_folds) - strict))
print("  of which less data       : %.4f" % (np.mean(full_folds) - np.mean(subsampled_folds)))
print("  of which leakage removed : %.4f" % (np.mean(subsampled_folds) - strict))

**Of the total 0.1134 drop, 0.0248 is having less training data and 0.0886 is leakage.**

So roughly **four fifths of the fall is real** and one fifth is an artefact of the strict split also being
a smaller one. The chapter's headline survives comfortably, and it is now quantified rather than asserted.

This is a habit worth generalising: **whenever a stricter procedure gives a worse number, check whether it
also gave the model less to learn from.** Subsampling the lenient split to match is two lines and it turns
"the honest number is lower" into "the honest number is lower, and here is how much of that is honesty".

### E12 - the dataset where a random split is fine

In [ ]:
one_per_member = (stacked.sample(frac=1, random_state=1)
                  .drop_duplicates("member_id").sort_index().reset_index(drop=True))
single_X = one_per_member[FEATURES]
single_y = one_per_member.target
single_groups = one_per_member.member_id


def forest_single(train_rows, test_rows):
    model = RandomForestClassifier(n_estimators=300, random_state=0, min_samples_leaf=1)
    model.fit(single_X.iloc[train_rows], single_y.iloc[train_rows])
    return roc_auc_score(single_y.iloc[test_rows],
                         model.predict_proba(single_X.iloc[test_rows])[:, 1])


random_mean = np.mean([forest_single(a, b) for a, b in
                       StratifiedKFold(5, shuffle=True, random_state=0).split(single_X, single_y)])
grouped_mean = np.mean([forest_single(a, b) for a, b in
                        GroupKFold(5).split(single_X, single_y, groups=single_groups)])

print("one row per member: %d rows, base rate %.4f" % (len(one_per_member), single_y.mean()))
print("  random split  %.4f" % random_mean)
print("  grouped split %.4f" % grouped_mean)
print("  inflation     %+.4f" % (random_mean - grouped_mean))
print()
print("for comparison, the stacked table's base rate is %.4f" % y.mean())

**The inflation vanishes** - it is **-0.0272**, which is to say the grouped split scored slightly
*higher*, well inside the fold-to-fold noise of 0.03 or so. With one row per member the two splitters are
doing almost the same thing, because grouping by an id that is unique per row is not a constraint at all.

That is the control this chapter needed: it confirms the inflation came from **repeated entities**, not
from something else about `GroupKFold`.

One detail worth noticing, because it is 04-01 returning uninvited: **the base rate changed, from 0.1198
to 0.2155.** Sampling one row per member weights every member equally, while the stacked table weights
members by how many rows they have - and members who cancel early have fewer rows. Row-weighting
systematically dilutes the positives. **Changing the unit of observation changed the answer**, exactly as
04-01's three churn rates did, and this time it happened as a side effect of an exercise about splitting.

### E11 - forward chaining as a splitter

In [ ]:
def forward_chaining_splits(frame, walls, test_window, group_mask):
    for wall in walls:
        train_rows = np.where((frame.cut_month <= wall) & ~group_mask)[0]
        test_rows = np.where((frame.cut_month > wall)
                             & (frame.cut_month <= wall + test_window) & group_mask)[0]
        if len(train_rows) and len(np.unique(y.iloc[test_rows])) == 2:
            yield train_rows, test_rows


def logistic_on(train_rows, test_rows):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(X.iloc[train_rows], y.iloc[train_rows])
    return roc_auc_score(y.iloc[test_rows], model.predict_proba(X.iloc[test_rows])[:, 1])


rows = []
for name, scorer in [("logistic regression", logistic_on), ("random forest", forest_on)]:
    folds = [scorer(a, b) for a, b in
             forward_chaining_splits(stacked, [8, 10, 12, 14, 16], 2, held_out_members)]
    rows.append({"model": name, "folds": len(folds), "mean": round(float(np.mean(folds)), 4),
                 "sd": round(float(np.std(folds)), 4),
                 "per fold": " ".join("%.3f" % f for f in folds)})
print(pd.DataFrame(rows).to_string(index=False))

The spread is around 0.044 for both - **larger than the 0.021 the leaky random-row folds showed**, and
that is the honest picture. Time periods differ from one another more than random samples of rows do,
because whatever changed between month 9 and month 17 changed for everybody at once.

**Reporting the mean alone would understate the uncertainty by half.** For anything that will run on
future data, the fold-to-fold spread of a forward-chained evaluation is a better guide to what next month
will bring than any single number.

## Interpretation

### E13

**The two splits they should have used:**

1. **Group by user.** With one row per user-item interaction, a random split trains on a user's other
   interactions and tests on this one - the chapter's exact bug, and recommenders are unusually
   vulnerable because a user's history *is* the signal. A user-grouped split would have shown how the
   system performs for people it has not seen.
2. **Split by time.** Recommendations are made forward: today's model must predict tomorrow's clicks. A
   random split lets it learn from next month's trends, which matters enormously when catalogues and
   fashions move.

**What each would have revealed:** the user-grouped split would expose how much of the 0.94 was
memorising individuals rather than learning preferences. The time split would expose how fast the model
goes stale, which is a different and equally important number - it tells you the retraining cadence.

For a production recommender you want both, and the honest offline number is usually far below 0.94.
A third possibility worth naming: **popularity bias** - if the offline metric rewards recommending
already-popular items, a baseline of "recommend the top 20 items" often scores close to the model, which
is 04-02's lesson rather than this chapter's.

### E14

**The argument for changing it:** the number currently reported is not an estimate of anything the
business experiences. The model runs monthly on the current membership, so the question is "how well will
it do next month?" and the evaluation answers "how well does it do on months it has already seen, for
members it has already met". Those differ here by up to a tenth of an AUC, and - worse - they can rank
models differently, so the last year of model choices may have been made on the wrong criterion.

**The argument somebody will make against:** "it has looked fine for a year, and the model performs
acceptably in production, so the evaluation is clearly good enough." This is not a stupid argument and it
deserves a real answer: the model may well be fine, because a leaky *evaluation* does not make a bad
model - it just fails to tell you when one is bad. The cost is not paid continuously; it is paid the day
somebody proposes a flexible new model that leaks better, wins the comparison, and is shipped.

**How to settle it cheaply:** compute both numbers for one month. If they agree, the concern is
theoretical and you have a two-line assertion to add. If they disagree by 0.10, the conversation is over.
Running the check is much cheaper than winning the argument.

## Debugging

### E15

1. **Check the group assignment actually did something**:
   `len(set(train.member_id) & set(test.member_id))` should be 0, and the number of distinct members per
   fold should be about a fifth of the total. A `GroupKFold` given the wrong column silently behaves like
   an ordinary `KFold`.
2. **Check the class balance per fold.** `GroupKFold` does not stratify, so with a rare class and few
   groups a fold can end up with very few positives, and an AUC on eight positives is nearly meaningless.
   If the folds are unbalanced, `StratifiedGroupKFold` exists.
3. **Check whether the drop is capacity-dependent.** Re-run with a depth-2 tree or a logistic regression.
   If the simple model barely moves while the flexible one collapses - exactly the pattern of this
   chapter - the new number is right and the old one was leaking. If *everything* collapses equally,
   suspect a bug in the new splitting code instead.

The colleague is wrong in a specific and forgivable way: **they are treating the higher number as the
baseline and the lower one as the deviation to be explained.** The burden runs the other way.

### E16

**Explanation 1: the folds genuinely differ.** Grouped folds partition *members*, and members are not
interchangeable - one fold may collect more short-tenured or more seasonal members. With 5 folds and 580
members, each fold's ~116 members carry real composition noise. **Diagnostic:** report the base rate and
the member characteristics per fold; if the high-scoring folds have a different positive rate or tenure
distribution, this is it. Increasing the number of folds, or repeating with different group assignments,
should shrink the spread.

**Explanation 2: too few positives per fold.** At a 12% base rate a fold might hold only a handful of
cancellers, and AUC on a handful is extremely noisy. **Diagnostic:** print `y.iloc[test].sum()` per fold.
If some folds have fewer than about 20 positives, the spread is a sample-size artefact and the fix is
`StratifiedGroupKFold` or fewer, larger folds.

The two are distinguishable and often both present. Note that the chapter's own grouped folds
(0.7129, 0.6180, 0.6424, 0.6668, 0.6717, sd 0.0317) are considerably more spread than the leaky random
folds (sd 0.0206) - which is not a defect of grouping but an honest measurement of variation the random
split was concealing.

## Exam and interview reasoning

### E17

**(a) 50,000 chest X-rays from 3,000 patients** - split by **patient**, stratified on the diagnosis. Time
matters only if practice or equipment changed over the collection period, in which case add a
chronological holdout.

**(b) Two years of daily store sales** - **chronological**, with forward chaining for validation. Rows are
independent in the entity sense (one row per day) but every prediction is forward, and seasonality means
the test period should ideally be a whole year or explicitly matched.

**(c) 100,000 independent loan applications** - a **stratified random split** is correct, provided
applications really are one-per-applicant. Check for repeat applicants first; if they exist, group.

**(d) 2 million clicks from 40,000 users over six months** - **grouped by user and chronological**, the
strictest combination. Both problems are present.

**The trap is (c)**, and it is a trap in two directions. It is the only one where a random split is right,
so it tests whether you apply the rules or recite them - but "independent loan applications" is exactly
the kind of phrase that turns out to be false on inspection, since the same person often applies more than
once and household members are correlated. **The answer that gets the job is "random split, after I have
checked that the applicant ids are unique."**

## Transfer to a different situation

### E18

**The split:** group by **card**, and split **chronologically**, with a **gap** between the training and
test periods - a gap of at least the labelling delay.

**The extra problem the delay creates:** at the moment you would train the model in production, **the most
recent 30-60 days of labels do not exist yet.** A model trained on data up to today has, in reality, only
been trained on labels up to two months ago. An evaluation that trains right up to the test boundary is
therefore optimistic about *label availability*, not about the split - and grouping and chronology both
leave it in place.

The fix is to make the offline setup match: train on data whose labels would have been mature at training
time, leaving a deliberate gap. Then the offline number reflects a model that could actually have existed
on that date. This is a real and expensive effect in fraud, and it is also why fraud models are often
evaluated on a rolling basis with an explicit "label maturity" cutoff.

Credit also for: bursts mean fraud is autocorrelated in time, so a random split within a burst is close to
duplicate leakage; and the base rate drifts, so the test period's base rate should be reported alongside
the metric.

## Explain it to someone non-technical

### E19

> "Imagine testing a new teacher by having them teach a class for a term and then quizzing the same
> students. They will do well - the teacher has learned exactly what these students respond to. That
> tells you nothing about how they would do with a class they have never met, which is the thing you
> actually want to know before hiring them. Our model was being tested on new months for members it had
> already studied for a year. When we tested it on members it had never seen, it did noticeably worse -
> and a different, simpler model turned out to be the better one."

(89 words.)

## Optional challenge

### E20 - the overlap that grouping and chronology do not fix

In [ ]:
test_rows = np.where((stacked.cut_month > 12) & held_out_members)[0]
rows = []
for gap in [0, 3, 6]:
    train_rows = np.where((stacked.cut_month <= 12 - gap) & ~held_out_members)[0]
    rows.append({"gap (months)": gap, "training rows": len(train_rows),
                 "AUC": round(forest_on(train_rows, test_rows), 4)})

# the control: the same shrunken training size, but drawn from the full pre-wall period
control_rng = np.random.default_rng(0)
available = np.where((stacked.cut_month <= 12) & ~held_out_members)[0]
control_rows = control_rng.choice(available, size=rows[-1]["training rows"], replace=False)
control = forest_on(control_rows, test_rows)

print(pd.DataFrame(rows).to_string(index=False))
print()
print("control: %d rows drawn from months 1-12 with NO gap -> AUC %.4f"
      % (len(control_rows), control))
print()
print("at gap 6: total fall from gap 0 is %.4f" % (rows[0]["AUC"] - rows[2]["AUC"]))
print("  of which less training data : %.4f" % (rows[0]["AUC"] - control))
print("  of which the removed overlap: %.4f" % (control - rows[2]["AUC"]))

**Yes, the score falls - and most of the fall is the training set shrinking, not the overlap.**

The raw numbers look dramatic: 0.6446 with no gap, 0.5564 with a six-month gap. But a six-month gap costs
1,990 of 2,740 training rows, and the control - the same 750 rows, drawn from the full pre-wall period
with no gap at all - scores 0.5932. So of the 0.0882 fall, about **0.0514 is simply having 750 rows
instead of 2,740**, and about **0.0368 is the overlap actually being removed**.

An 0.037 leak is smaller than the 0.096 from repeated members, and it is real, and **neither grouping nor
a chronological wall removes it.** The mechanism is worth stating precisely, because it is the subtlest
thing in this module:

> A prediction made in month 12 asks about months 13-18. A prediction made in month 8 asks about months
> 9-14. **Those two targets share five months of the same future.** So a training row from month 8 and a
> test row from month 13 are partly answering the same question, even though the members are disjoint and
> the prediction dates are properly ordered.

This is called **target overlap** or, in finance, an overlapping-samples problem, and the standard fix is
the one measured here: **leave a gap of at least one horizon between the last training row and the first
test row.** It costs data, which is why you should measure the trade rather than assume it - here, buying
0.037 of honesty cost 0.051 of performance, and on a larger dataset the second term would shrink while the
first would not.

The general lesson to take out of module 04's splitting chapters: **a split is a claim about what the
model will not know at prediction time.** Every leak in these two chapters was a place where that claim
was false, and each was found by asking the same question - *could this row have known the answer?*